# 00 · Access tests (spec §3)
T1–T6 as specified. The live code is `scripts/access_tests.py`; set `RUN_LIVE=True` to re-run it (≈10 min + downloads). With `RUN_LIVE=False` this notebook shows the measurements recorded by that script on **2026-09-25** from the development laptop (link ≈1.7 MB/s, measured against speed.cloudflare.com).

In [1]:
RUN_LIVE = False
import json, subprocess, sys, pandas as pd
if RUN_LIVE:
    subprocess.run([sys.executable, '../scripts/access_tests.py'], cwd='..', check=True)
res = json.load(open('../data/access_tests.json'))
list(res)

['T1', 'T2', 'T3', 'T4', 'T5', 'T6']

## Summary

In [2]:
rows = []
for k in ['T1','T2','T3','T5','T6']:
    r = res[k]; rows.append(dict(test=k, ok=r.get('ok'), detail={kk: vv for kk, vv in r.items() if kk not in ('ok','vars','leads')}))
rows.append(dict(test='T4', ok=res['T4']['imd']['ok'], detail=res['T4']['imd']))
pd.set_option('display.max_colwidth', 300)
pd.DataFrame(rows)

,test,ok,detail
0,T1,True,"{'shape': [161, 201], 'open_s': 10.0, 'total_s': 13.8, 'units': None, 'max_value': 0.53948974609375, 'mean_value': 0.006227322854101658, 'chunks': [1, 1, 721, 1440], 'dims': ['time', 'prediction_timedelta', 'latitude', 'longitude'], 'n_time': 5134}"
1,T2,True,"{'sizes': {'number': 50, 'longitude': 33, 'latitude': 27}, 'total_s': 35.9, 'units': None, 'chunks': [1, 50, 8, 240, 121], 'dims': ['time', 'number', 'prediction_timedelta', 'longitude', 'latitude'], 'time_range': ['2018-01-01T00:00:00.000000000', '2022-12-31T12:00:00.000000000'], 'n_time': 3652}"
2,T3,True,"{'sizes': {'time': 366, 'lat': 129, 'lon': 135}, 'download_s': 0.4, 'lat': [6.5, 38.5], 'lon': [66.5, 100.0], 'land_cells': 4964, 'jul15_india_mean_mm': 8.446549809457528}"
3,T5,None,{'note': 'Phase 2 only; endpoint reachable (HTTP 200); retrieval needs ECMWF account - not attempted'}
4,T6,True,"{'hres_s_per_init_10leads': 8.31, 'ens_s_per_init_10leads': 105.21, 'hres_MB': 1.29, 'ens_MB': 1.78, 'est_hours_2018_2022_serial': 115.1}"
5,T4,True,"{'ok': True, 'bytes': 239850, 'n': 36, 'crs': 'EPSG:4326', 'cols': ['OBJECTID_1', 'region_cod', 'subdivisio', 'STATE', 'Area', 'geometry'], 'bounds': [-796555.47, 764107.03, 2128753.72, 4116495.93], 's': 1.1}"


## Findings
* **T1** HRES: `total_precipitation_24hr` in metres (no `units` attr); chunk = one global field per (init, lead). No `total_column_water_vapour` → state features use `specific_humidity` 850 hPa.
* **T2** ENS 1.5°: 50 members, 2018-01-01 → 2022-12-31T12. Chunk = (1 init, 50 members, 8 × 6-h leads, global) ≈ 40 MB → Day 1–10 needs 6 chunks ≈ 240 MB per init.
* **T3** IMD: `imdlib.get_data` hung on this network (requests over NAT64); the same official endpoint works with `curl -X POST -d rain=YYYY`. `imdlib.open_data` reads the files.
* **T4** IMD `sd_boundary.json`: 36 polygons; declares EPSG:4326 but coordinates are UTM 44N metres → `set_crs(32644, allow_override=True)`. A & N Islands and Lakshadweep have **no IMD land cells** → no truth.
* **T5** Phase 2 only, not attempted.
* **T6** HRES ≈ 8 s/init, ENS ≈ 105 s/init at ≈1.9 MB/s. Full 2018–2022 (3 652 inits) ≈ 876 GB ENS reads ≈ 115 h serial on this link → full extraction must run on Colab / a GCP VM (spec §2).